# RAG — Naive Baseline

Goal: implement the simplest possible retrieve-then-generate pipeline and measure answer quality.

In [ ]:
import sys
sys.path.insert(0, '../..')

from src.client import get_client

client = get_client()
MODEL = 'claude-sonnet-4-6'

In [ ]:
# --- toy corpus ---
corpus = [
    {"id": 0, "text": "Retrieval-Augmented Generation (RAG) combines a retriever with a language model."},
    {"id": 1, "text": "Dense retrieval uses embedding similarity; sparse retrieval uses BM25."},
    {"id": 2, "text": "Re-ranking improves precision by scoring candidate passages with a cross-encoder."},
]

def retrieve(query: str, k: int = 2) -> list[str]:
    """Keyword overlap retrieval — intentionally naive."""
    q_tokens = set(query.lower().split())
    scored = [
        (len(q_tokens & set(d["text"].lower().split())), d["text"])
        for d in corpus
    ]
    scored.sort(reverse=True)
    return [text for _, text in scored[:k]]


def answer(query: str) -> str:
    chunks = retrieve(query)
    context = '\n'.join(f'- {c}' for c in chunks)
    resp = client.messages.create(
        model=MODEL,
        max_tokens=256,
        messages=[{"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}],
    )
    return resp.content[0].text


print(answer("What is RAG?"))